In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

SEED = 214

# Feature Engineering for Human Preference Prediction

Goal: construct interpretable pairwise features from Prompt, Response A and Response B, and evaluate whether they contain predictive signals for human preference.

In [2]:
DATA_PATH = Path("../data/train.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(57477, 9)


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [3]:
#merge the 3 one-hot code into 1 label (A/B/tie)
df["label"] = np.select(
    [
        df["winner_model_a"] == 1,
        df["winner_model_b"] == 1,
        df["winner_tie"] == 1
    ],
    [
        "A",
        "B",
        "tie"
    ]
)

In [4]:
df["prompt_list"] = df["prompt"].apply(json.loads)
df["response_a_list"] = df["response_a"].apply(json.loads)
df["response_b_list"] = df["response_b"].apply(json.loads)

# Feature: length of the text (not accounting for the turns)

In [5]:
def total_char_length(turns):
    return sum(len(turn) for turn in turns)

In [6]:
df["prompt_chars"] = df["prompt_list"].apply(total_char_length)
df["a_chars"] = df["response_a_list"].apply(total_char_length)
df["b_chars"] = df["response_b_list"].apply(total_char_length)

TypeError: object of type 'NoneType' has no len()

### Nested Missing Responses

During response-length feature construction, a small number of nested `None` values were identified inside parsed response lists. These missing turns were further examined before defining the length features.

In [7]:
#check the number of "None" in both response seperately
a_has_none = df["response_a_list"].apply(
    lambda turns: any(turn is None for turn in turns)
)

b_has_none = df["response_b_list"].apply(
    lambda turns: any(turn is None for turn in turns)
)

p_has_none = df["prompt_list"].apply(
    lambda turns: any(turn is None for turn in turns)
)

print("Prompt contains None:", p_has_none.sum())
print("Response A contains None:", a_has_none.sum())
print("Response B contains None:", b_has_none.sum())

Prompt contains None: 0
Response A contains None: 111
Response B contains None: 120


In [8]:
df.loc[
    a_has_none,
    ["prompt_list", "response_a_list", "response_b_list", "label"]
].head()

,prompt_list,response_a_list,response_b_list,label
207,"[Tell me about wwiii, Enough]",[I do not have any confirmed details about a W...,"[As an AI language model, I must clarify that ...",B
385,[How does stochastic gradient Langevin dynamic...,[Stochastic Gradient Langevin Dynamics (SGLD) ...,[Stochastic Gradient Langevin Dynamics (SGLD) ...,A
780,[I want you to act as if you are a classic tex...,[None],[None],tie
1763,"[Create a game, Code, Create a game with the c...","[Sure, I'd be happy to help you create a game!...","[Sure, I'd be happy to help you create a game!...",B
2530,[Please help me prepare for an interview with ...,[Sure! Here are some potential questions you m...,"[Sure, I'd be happy to help you prepare for yo...",tie


In [9]:
both_none = a_has_none & b_has_none
only_a_none = a_has_none & ~b_has_none
only_b_none = b_has_none & ~a_has_none

print("Both contain None:", both_none.sum())
print("Only A contains None:", only_a_none.sum())
print("Only B contains None:", only_b_none.sum())

Both contain None: 70
Only A contains None: 41
Only B contains None: 50


In [10]:
print(df.loc[only_a_none, "label"].value_counts(normalize=True))
print(df.loc[only_b_none, "label"].value_counts(normalize=True))
print(df.loc[both_none, "label"].value_counts(normalize=True))
#missing responses may affect preference, especially when only B contains None
#but the sample size is small and the same pattern is not clearly observed when only A contains None

B      0.365854
tie    0.341463
A      0.292683
Name: label, dtype: float64
A      0.52
B      0.26
tie    0.22
Name: label, dtype: float64
tie    0.357143
B      0.328571
A      0.314286
Name: label, dtype: float64


In [24]:
df["a_missing_turns"] = df["response_a_list"].apply(
    lambda turns: sum(turn is None for turn in turns)
)

df["b_missing_turns"] = df["response_b_list"].apply(
    lambda turns: sum(turn is None for turn in turns)
)

df["a_missing_ratio"] = (
    df["a_missing_turns"] / df["response_a_list"].apply(len)
)

df["b_missing_ratio"] = (
    df["b_missing_turns"] / df["response_b_list"].apply(len)
)

df.loc[
    (df["a_missing_turns"] > 0) | (df["b_missing_turns"] > 0),
    [
        "label",
        "a_missing_turns",
        "b_missing_turns",
        "a_missing_ratio",
        "b_missing_ratio"
    ]
].head(20)

,label,a_missing_turns,b_missing_turns,a_missing_ratio,b_missing_ratio
207,B,1,0,0.500000,0.000000
385,A,1,0,0.125000,0.000000
763,A,0,1,0.000000,0.200000
780,tie,1,1,1.000000,1.000000
1763,B,1,1,0.333333,0.333333
2530,tie,1,0,0.250000,0.000000
2648,B,1,0,0.250000,0.000000
2770,tie,0,1,0.000000,0.333333
2989,A,0,1,0.000000,0.250000
3152,A,1,1,0.500000,0.500000


In [26]:
a_all_missing = df["response_a_list"].apply(
    lambda turns: all(turn is None for turn in turns)
)

b_all_missing = df["response_b_list"].apply(
    lambda turns: all(turn is None for turn in turns)
)

print("A all missing:", a_all_missing.sum())
print("B all missing:", b_all_missing.sum())

print(df.loc[a_all_missing & ~b_all_missing, "label"].value_counts(normalize=True))
print(df.loc[b_all_missing & ~a_all_missing, "label"].value_counts(normalize=True))

both_all_missing = a_all_missing & b_all_missing

print("Both all missing:", both_all_missing.sum())


A all missing: 19
B all missing: 19
Series([], Name: label, dtype: float64)
Series([], Name: label, dtype: float64)
Both all missing: 19


In [27]:
df.loc[
    both_all_missing,
    "label"
].value_counts(normalize=True)

tie    0.631579
B      0.210526
A      0.157895
Name: label, dtype: float64

In [28]:
def missing_positions(turns):
    return [
        i for i, turn in enumerate(turns)
        if turn is None
    ]

df["a_missing_positions"] = df["response_a_list"].apply(missing_positions)
df["b_missing_positions"] = df["response_b_list"].apply(missing_positions)

missing_samples = df[
    (df["a_missing_turns"] > 0) |
    (df["b_missing_turns"] > 0)
]

missing_samples[
    [
        "a_missing_turns",
        "b_missing_turns",
        "a_missing_positions",
        "b_missing_positions",
        "label"
    ]
].head(30)

,a_missing_turns,b_missing_turns,a_missing_positions,b_missing_positions,label
207,1,0,[1],[],B
385,1,0,[6],[],A
763,0,1,[],[2],A
780,1,1,[0],[0],tie
1763,1,1,[2],[2],B
2530,1,0,[2],[],tie
2648,1,0,[2],[],B
2770,0,1,[],[1],tie
2989,0,1,[],[1],A
3152,1,1,[1],[1],A


In [31]:
same_missing_positions = (
    missing_samples["a_missing_positions"]
    ==
    missing_samples["b_missing_positions"]
)

print(
    "Same missing positions ratio:",
    same_missing_positions.mean()
)

Same missing positions ratio: 0.43478260869565216
